<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# Bloque 08 · Excel avanzado con openpyxl

Caso del Observatorio documental. Ejecuta las celdas en orden desde la carpeta del bloque. Lee la explicación, modifica un dato del ejemplo y comprueba cómo cambia el resultado. Los archivos generados quedan en una carpeta temporal.

Instala las dependencias con `%pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporal = TemporaryDirectory()
salida = Path(temporal.name)
ejemplos = Path("ejemplos")
from openpyxl import Workbook, load_workbook
from openpyxl.formula.translate import Translator
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.formatting.rule import FormulaRule, CellIsRule, DataBarRule
from openpyxl.styles import PatternFill
from openpyxl.chart import BarChart, Reference
from collections import Counter
from excel_revision import leer_entrada, construir

## Ejemplo 1 · Contrato y granularidad

Cada fila es una entrada. La cabecera fija el orden de los campos y la validación detecta identificadores repetidos, tamaños inválidos y estados desconocidos antes de construir el libro. Contrastamos la distribución en Python para disponer de una referencia independiente de Excel.

In [ ]:
datos = leer_entrada(ejemplos / "seguimiento_entrada.xlsx")
print(datos[0])
print(Counter(fila[4] for fila in datos))
assert len(datos) == 10

## Ejemplo 2 · Referencias y fórmulas

Una referencia relativa se desplaza al copiar. El signo $ fija su columna o su fila. Translator mueve referencias A1, pero el cálculo queda a cargo de Excel. En el archivo, usa nombres ingleses de funciones y comas entre argumentos.

In [ ]:
libro = Workbook()
hoja = libro.active
hoja.title = "Control"
hoja.append(["Documento", "Bytes", "Estado", "Alerta"])
hoja.append(["DOC-001", 25, "Pendiente"])
hoja.append(["DOC-002", 40, "Revisado"])
hoja["H1"] = 30
base = '=IF(B2>$H$1,1,0)'
hoja["D2"] = base
hoja["D3"] = Translator(base, origin="D2").translate_formula("D3")
print(hoja["D3"].value)
print(Translator("=$B2+C$1", origin="D2").translate_formula("E3"))

## Ejemplo 3 · Tabla, autofiltro y crecimiento

Un autofiltro de hoja se aplica a un rango. Una tabla además tiene nombre, estilo y perímetro propios. Al añadir filas, actualiza la tabla, su filtro y los rangos dependientes. Construirlos a partir de la última fila evita que queden registros fuera.

In [ ]:
hoja.auto_filter.ref = "A1:B3"
print("Filtro independiente:", hoja.auto_filter.ref)
hoja.auto_filter.ref = None
tabla = Table(displayName="ControlDocs", ref="A1:D3")
tabla.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
hoja.add_table(tabla)
hoja.append(["DOC-003", 31, "Pendiente"])
ultima = hoja.max_row
hoja[f"D{ultima}"] = Translator(base, origin="D2").translate_formula(f"D{ultima}")
tabla.ref = f"A1:D{ultima}"
hoja["H2"] = f"=SUM(D2:D{ultima})"

## Ejemplo 4 · Validaciones de lista y número

Una lista corta puede escribirse directamente. Si el catálogo se mantiene aparte, usa un rango con nombre. Excel aplica las restricciones durante la edición, mientras Python debe validar sus propios datos de entrada.

In [ ]:
catalogo = libro.create_sheet("Catalogo")
for estado in ("Pendiente", "Revisado"):
    catalogo.append([estado])
libro.defined_names.add(DefinedName("EstadosRevision", attr_text="'Catalogo'!$A$1:$A$2"))
lista = DataValidation(type="list", formula1="EstadosRevision", showErrorMessage=True)
hoja.add_data_validation(lista)
lista.add(f"C2:C{ultima}")
limite = DataValidation(type="whole", operator="greaterThanOrEqual", formula1="0", showErrorMessage=True)
hoja.add_data_validation(limite)
limite.add(f"B2:B{ultima}")

## Ejemplo 5 · Formato condicional y prioridades

La fórmula se interpreta desde la esquina superior izquierda del rango. $C2 mira siempre la columna C de la fila actual. Cuando dos reglas coinciden, la prioridad y stopIfTrue controlan qué reglas siguen evaluándose.

In [ ]:
ambar = PatternFill("solid", fgColor="FFFFCC66")
rojo = PatternFill("solid", fgColor="FFFFAAAA")
hoja.conditional_formatting.add(f"A2:D{ultima}", FormulaRule(formula=['$C2="Pendiente"'], fill=ambar, stopIfTrue=True))
hoja.conditional_formatting.add(f"A2:D{ultima}", FormulaRule(formula=['$B2>30'], fill=rojo))
hoja.conditional_formatting.add(f"B2:B{ultima}", DataBarRule(start_type="min", end_type="max", color="FF338899"))
for rango in hoja.conditional_formatting:
    print(rango.sqref, [(r.priority, r.stopIfTrue) for r in hoja.conditional_formatting[rango]])

## Ejemplo 6 · Resumen, gráfico y reapertura

Las barras comparan estados. Las categorías excluyen la cabecera y la serie la usa como título. El resumen conserva cifras exactas. Tras guardar, se verifica la estructura con openpyxl y el aspecto o recálculo se puede revisar en Excel.

In [ ]:
resumen = libro.create_sheet("Resumen")
resumen.append(["Estado", "Cantidad"])
for estado, cantidad in Counter(c.value for c in hoja["C"][1:]).items():
    resumen.append([estado, cantidad])
grafico = BarChart()
grafico.title = "Documentos por estado"
grafico.add_data(Reference(resumen, min_col=2, min_row=1, max_row=3), titles_from_data=True)
grafico.set_categories(Reference(resumen, min_col=1, min_row=2, max_row=3))
resumen.add_chart(grafico, "D2")
ruta = salida / "demo.xlsx"
libro.save(ruta)
with_formula = load_workbook(ruta)
with_cache = load_workbook(ruta, data_only=True)
assert with_formula["Control"].tables["ControlDocs"].ref == "A1:D4"
print(with_formula["Control"]["D2"].value, with_cache["Control"]["D2"].value)
with_formula.close()
with_cache.close()

## Ejemplo 7 · Integración del observatorio

El generador de clase reúne la tabla, el estado manual, la fórmula de alerta y el resumen. La aceptación comprueba filas y pendientes con los datos originales, antes del recálculo de Excel.

In [ ]:
control = construir(datos)
revision = control["Revision"]
assert revision.max_row == len(datos) + 1
pendientes = sum(c.value == "Pendiente" for c in revision["G"][1:])
print("Pendientes:", pendientes, "de", len(datos))
control.save(salida / "revision.xlsx")

## Diez ejercicios

Completa las celdas TODO. Cada enunciado incluye una comprobación para validar tu resultado.

### Ejercicio 01 · Traducir referencias mixtas

Traslada =$B2+C$1 de D2 a F5 y guarda el resultado en formula.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert formula == "=$B5+E$1"

### Ejercicio 02 · Fórmula frente a caché

Reabre demo.xlsx de ambas formas y guarda D3 como expresion y cache. Cierra los libros.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert expresion == "=IF(B3>$H$1,1,0)" and cache is None

### Ejercicio 03 · Reconciliar los estados

Cuenta las entradas por estado original y guarda el Counter en conteos.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert sum(conteos.values()) == 10

### Ejercicio 04 · Ampliar una tabla y su filtro

Sobre una copia abierta de demo.xlsx añade DOC-004 de 32 bytes como Revisado. Ajusta fórmula, tabla, autofiltro de tabla y suma en H2. Guarda la hoja en ampliada.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert ampliada.tables["ControlDocs"].autoFilter.ref == "A1:D5" and ampliada["H2"].value == "=SUM(D2:D5)"

### Ejercicio 05 · Añadir una opción al catálogo

Sobre un libro independiente incorpora En curso al catálogo y actualiza el rango definido. Guarda el libro en opciones.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert opciones.defined_names["EstadosRevision"].destinations.__next__() == ("Catalogo", "$A$1:$A$3")

### Ejercicio 06 · Limitar los tamaños

Define en restriccion una validación de números enteros entre 0 y 1024 para B2:B5.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert restriccion.formula2 == "1024" and "B5" in restriccion

### Ejercicio 07 · Resaltar alertas

Crea en regla_alerta una regla que pinta la fila cuando D vale 1. Usa referencia mixta.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert regla_alerta.formula == ["$D2=1"]

### Ejercicio 08 · Construir otra serie

Construye en barras un gráfico basado en Resumen con los dos estados del ejemplo.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert barras.series[0].val.numRef.f.endswith("$B$2:$B$3")

### Ejercicio 09 · Validar una entrada incorrecta

Copia los datos al formato de entrada y altera el tamaño de la primera fila a −1. Guarda en temporal y comprueba que leer_entrada lo rechaza. Guarda el mensaje en diagnostico.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert "inválido" in diagnostico

### Ejercicio 10 · Publicar y verificar

Construye revision_final.xlsx y reábrelo. Guarda en verificado el libro. Comprueba tabla, total y validación de la última fila.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert verificado["Revision"].tables["Documentos"].ref == "A1:H11"
# assert verificado["Resumen"]["B10"].value == "=SUM(B2:B8)"
# assert "G11" in verificado["Revision"].data_validations.dataValidation[0]